In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 토크나이저 및 모델

## Lua 언어 특화 토크나이저 선정
15b와 1b가 있지만, local에서 사용하기 위해 경량화된 1b 모델을 사용
 - https://huggingface.co/nuprl/MultiPL-T-StarCoderBase_1b
 - https://huggingface.co/nuprl/MultiPL-T-StarCoderBase_15b

## 토크나이저와 모델 import

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

# colab에서 세션이 꺼지면 다운 받았던 모델을 다시 다운받아야해서, cache를 drive에 저장하는 방법
os.environ["TRANSFORMERS_CACHE"] = "/content/drive/MyDrive/huggingface_cache"

model_name = "nuprl/MultiPLCoder-1b"

tokenizer = AutoTokenizer.from_pretrained(model_name)
lua_revision = "7e96d931547e342ad0661cdd91236fe4ccf52545"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    revision=lua_revision,
    torch_dtype="auto",
    device_map="auto"
).cuda()


_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

위 오류를 수정하기 위해 아래 코드 추가.


In [ ]:
import torch
# 모델의 모든 파라미터 및 연산 데이터 타입을 32비트에서 16비트 부동소수점으로 변환하는 역할
model = model.to(torch.float16)


## PEFT 기법 적용
- 전체 모델을 파인튜닝하는 것보다 훨씬 적은 계산 비용과 메모리로 원하는 과제에 훈련 가능
- 프롬프트 튜닝, 어댑터, LoRA 등의 방법이 있음.


In [ ]:
!pip install peft

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    task_type="CAUSAL_LM",  # 텍스트 생성(코드 완성) 작업 유형
    inference_mode=False,   # 훈련 모드로 설정, True로 하면 추론모드로 설정 된다.
    target_modules=["c_fc", "c_attn", "c_proj"],
    r=16,                    # Low-rank 행렬의 차원. (보통 8, 16, 32 사용)
    lora_alpha=32,          # 스케일링 요소. 보통 r의 2배 또는 4배로 설정.
    lora_dropout=0.1,       # LoRA 레이어에 적용할 드롭아웃 비율
)

model = get_peft_model(model, lora_config)

In [ ]:
model.print_trainable_parameters()

trainable params: 11,108,352 || all params: 1,148,315,648 || trainable%: 0.9674


In [ ]:
tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # 입력 시퀀스의 길이를 맞추기 위한 특수 토큰
model.resize_token_embeddings(len(tokenizer))

Embedding(49153, 2048)

## 데이터 콜레이터 지정
- 자동으로 각 배치마다 입력 길이가 다를 때 패딩을 맞춰준다.
- 학습에 필요한 입력값과 정답 레이블을 생성한다.

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Lua 함수 자동완성 함수

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList

# 해당 문자가 나왔을 때 더 이상 생성하지 않도록 하는 규칙
class StopOnSequences(StoppingCriteria):
    def __init__(self, stop_sequences, tokenizer):
        self.stop_ids = [tokenizer.encode(s, add_special_tokens=False) for s in stop_sequences]

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        seq = input_ids[0].tolist()
        for ids in self.stop_ids:
            if len(seq) >= len(ids) and seq[-len(ids):] == ids:
                return True
        return False

stop_words = ["\n\n", "print", "--", "function", "return"]


def autocomplete_lua_function(prompt: str) -> str:
    stops = StoppingCriteriaList([StopOnSequences(stop_words, tokenizer)])

    # 모델에 입력
    output = model.generate(
        tokenizer(prompt, return_tensors="pt").input_ids.cuda(),
        max_new_tokens=50,
        do_sample=False,
        temperature=0.4,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
        stopping_criteria=stops,
        repetition_penalty=1.2,
    )

    # 토큰 → 텍스트
    generated_code = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"generated_code : \n{generated_code}\n")

    # 프롬프트 이후 부분만 추출
    generated_code = generated_code[len(prompt):] # 프롬프트 길이 만큼은 사용하지 않는다.
    # print(f"generated_code[len(prompt):] : \n{generated_code}\n")

    # 첫 번째 함수 단위만 가져오기 (Lua: function ... end)
    # 설정 중 하나일 뿐, 제거해도 되는 설정이다.
    if "function" in generated_code:
        functions = generated_code.split("function")
        if len(functions) > 1:
            generated_code = "function" + functions[1].split("end")[0] + "end"

    # print(f"generated_code before return : \n{generated_code}\n")
    return generated_code.strip()


In [ ]:
# 학습 전 답변
print(autocomplete_lua_function("function add(a,b)\n return a + "))


generated_code : 
function add(a,b)
 return a +  b end

b end


In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


# 학습

## Trainer용 학습 설정

이 설정에 따라 ./multiplcoder-1b-lua-finetuned 폴더에
pytorch_model.bin (가중치), config.json, tokenizer_config.json 등이 저장됨

save_steps 속성에 따라 중간 단계 체크포인트도 같은 디렉터리에 저장

만약 따로 저장을 안했다면 다음과 같이 저장 가능
```
trainer.save_model("./saved_model_dir")
tokenizer.save_pretrained("./saved_model_dir")
```

여기서 토크나이저도 같이 저장하는 이유는 모델 배포 및 재사용 편의성에 있다. 학습은 모델이 하는 것이니 model만 저장해도 상관이없다.

불러올 때는 다음과 같이 사용한다
```
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("./saved_model_dir")
tokenizer = AutoTokenizer.from_pretrained("./saved_model_dir")
```


In [ ]:
from datasets import load_dataset

# "text" -> "json" 으로 변경, data_files에 jsonl 파일 경로 지정
dataset = load_dataset("json", data_files={"train": "./1_train_data/*.jsonl"}) #

def tokenize_function(examples):
    # prompt와 completion을 합쳐서 하나의 텍스트로 만듭니다.
    inputs = [p + c for p, c in zip(examples["prompt"], examples["completion"])]

    # 합쳐진 텍스트를 토큰화합니다.
    # max_length는 데이터 길이에 맞춰 적절히 조정하세요.
    return tokenizer(inputs, truncation=True, max_length=512)

# 토큰화 적용
tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
print(tokenized_dataset['train'][0])
empty_count = sum([len(x["input_ids"]) == 0 for x in tokenized_dataset["train"]])
print("빈 시퀀스 개수:", empty_count)
tokenized_dataset = tokenized_dataset.filter(lambda x: len(x["input_ids"]) > 0)

{'prompt': '[Sync]\nnumber DetectDistance = 4', 'completion': '', 'input_ids': [77, 4764, 79, 203, 2171, 25967, 8457, 280, 225, 38], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
빈 시퀀스 개수: 0


In [ ]:
train_test = tokenized_dataset['train'].train_test_split(test_size=0.2, seed = 121)

print(train_test)

# train, test로 분리된 버전 사용
train_dataset = train_test['train']
eval_dataset = train_test['test']

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion', 'input_ids', 'attention_mask'],
        num_rows: 816
    })
    test: Dataset({
        features: ['prompt', 'completion', 'input_ids', 'attention_mask'],
        num_rows: 204
    })
})


### 하이퍼파라미터 튜닝

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./multiplcoder-1b-lua-finetuned",
    per_device_train_batch_size=1, # 데이터 변경 후 1 -> 2
    gradient_accumulation_steps=4, # 데이터 변경 후 4 -> 2
    max_steps=500,# 데이터 변경 후 200 -> 500
    save_steps=200,
    save_total_limit=2,
    num_train_epochs=3, # 데이터 변경 후 파라미터 추가
    weight_decay=0.0, # 파라미터 변경 후 0 -> 0.01
    warmup_ratio=0.03,
    logging_steps=10,
    bf16=False,
    fp16=True,             # 지원 GPU 시
    optim="adamw_torch",
    report_to=[],
    learning_rate=1.2e-4,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [ ]:
import math

print("--- [수동 평가] 학습 전 성능 측정 ---")
metrics_before_training = trainer.evaluate()
try:
    perplexity = math.exp(metrics_before_training["eval_loss"])
    print(f"Perplexity before training: {perplexity:.2f}")
except KeyError:
    print("Could not calculate perplexity. Raw metrics:")
print(metrics_before_training)

# 이전 모델 : Qwen/Qwen2.5-Coder-1.5B-Instruct
# --- [수동 평가] 학습 전 성능 측정 ---
#  [26/26 00:02]
# Perplexity before training: 94.68
# {'eval_loss': 4.550539016723633, 'eval_model_preparation_time': 0.0177, 'eval_runtime': 2.1041, 'eval_samples_per_second': 96.956, 'eval_steps_per_second': 12.357}

# 모델 변경 nuprl/MultiPLCoder-1b
# --- [수동 평가] 학습 전 성능 측정 ---
#  [26/26 00:00]
# Perplexity before training: 46.13
# {'eval_loss': 3.831555128097534, 'eval_model_preparation_time': 0.0039, 'eval_runtime': 1.0535, 'eval_samples_per_second': 193.633, 'eval_steps_per_second': 24.679}

--- [수동 평가] 학습 전 성능 측정 ---


Perplexity before training: 46.14
{'eval_loss': 3.831693649291992, 'eval_model_preparation_time': 0.0106, 'eval_runtime': 1.2953, 'eval_samples_per_second': 157.496, 'eval_steps_per_second': 20.073}


## 모델 학습 시작

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 49152}.


Step,Training Loss
10,3.601500
20,3.363500
30,3.401600
40,2.695900
50,2.452000
60,2.593800
70,2.666900
80,2.331400
90,2.454900
100,2.161400


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:300: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:300: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:300: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


TrainOutput(global_step=500, training_loss=1.877316921234131, metrics={'train_runtime': 228.5948, 'train_samples_per_second': 8.749, 'train_steps_per_second': 2.187, 'total_flos': 255846714753024.0, 'train_loss': 1.877316921234131, 'epoch': 2.450980392156863})

In [ ]:
import math

print("--- [수동 평가] 학습 후 성능 측정 ---")
metrics_after_training = trainer.evaluate()
try:
    perplexity = math.exp(metrics_after_training["eval_loss"])
    print(f"Perplexity before training: {perplexity:.2f}")
except KeyError:
    print("Could not calculate perplexity. Raw metrics:")
print(metrics_after_training)


--- [수동 평가] 학습 후 성능 측정 ---


Perplexity before training: 5.34
{'eval_loss': 1.676160454750061, 'eval_model_preparation_time': 0.0106, 'eval_runtime': 1.2788, 'eval_samples_per_second': 159.524, 'eval_steps_per_second': 20.331, 'epoch': 2.450980392156863}


### 성능 기록
```

#### Qwen/Qwen2.5-Coder-1.5B-Instruct ####
# 바꾸고 후 성능
# --- [수동 평가] 학습 후 성능 측정 ---
#  [26/26 06:46]
# Perplexity before training: 17.19
# {'eval_loss': 2.8440983295440674, 'eval_model_preparation_time': 0.0177, 'eval_runtime': 2.0193, 'eval_samples_per_second': 101.026, 'eval_steps_per_second': 12.876, 'epoch': 2.450980392156863}

# 학습률 1e-4 에서 8e-5 로 낮췃을 때
# --- [수동 평가] 학습 후 성능 측정 ---
#  [26/26 07:23]
# Perplexity before training: 19.74
# {'eval_loss': 2.9828600883483887, 'eval_model_preparation_time': 0.012, 'eval_runtime': 1.9318, 'eval_samples_per_second': 105.602, 'eval_steps_per_second': 13.459, 'epoch': 2.450980392156863}

# 학습률 1.5e-4 로 올렷을 때
# --- [수동 평가] 학습 후 성능 측정 ---
#  [26/26 06:27]
# Perplexity before training: 13.28
# {'eval_loss': 2.5861589908599854, 'eval_model_preparation_time': 0.0124, 'eval_runtime': 1.9366, 'eval_samples_per_second': 105.337, 'eval_steps_per_second': 13.425, 'epoch': 2.450980392156863}

#### nuprl/MultiPLCoder-1b ####
# 모델 바꿨을 때 학습률 : 1.5e-4
# --- [수동 평가] 학습 후 성능 측정 ---
#  [26/26 03:56]
# Perplexity before training: 5.25
# {'eval_loss': 1.6590495109558105, 'eval_model_preparation_time': 0.0107, 'eval_runtime': 1.2653, 'eval_samples_per_second': 161.227, 'eval_steps_per_second': 20.549, 'epoch': 2.450980392156863}

# 모델 바꿨을 때 학습률 : 1e-4
# --- [수동 평가] 학습 후 성능 측정 ---
#  [26/26 04:12]
# Perplexity before training: 5.44
# {'eval_loss': 1.6929738521575928, 'eval_model_preparation_time': 0.0105, 'eval_runtime': 1.3103, 'eval_samples_per_second': 155.693, 'eval_steps_per_second': 19.843, 'epoch': 2.450980392156863}

# 모델 바꿨을 때 학습률 : 3e-5
# --- [수동 평가] 학습 후 성능 측정 ---
#  [26/26 04:42]
# Perplexity before training: 7.27
# {'eval_loss': 1.983271837234497, 'eval_model_preparation_time': 0.0107, 'eval_runtime': 1.2989, 'eval_samples_per_second': 157.056, 'eval_steps_per_second': 20.017, 'epoch': 2.450980392156863}

# 1e-4 부터는 뭔가 없는 함수들을 나타냄.
#  1.2e-4로 변경
# --- [수동 평가] 학습 후 성능 측정 ---
#  [26/26 05:13]
# Perplexity before training: 5.34
# {'eval_loss': 1.676160454750061, 'eval_model_preparation_time': 0.0106, 'eval_runtime': 1.2788, 'eval_samples_per_second': 159.524, 'eval_steps_per_second': 20.331, 'epoch': 2.450980392156863}
```

# 학습된 모델로 재답변 후 수정

코드 구조와 사용법은 바꾸지 않아도 되고, 새 모델 가중치를 사용해서 같은 API로 동작하게 하면 된다.

```
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
generated_code :
local currentTargetEntity = self.Entity.AIChaseComponent.AIChaseComponent.AIChaseComponent.AIChaseComponent.Entity.AIChaseComponent.AIChaseComponent.AIChaseComponent.AIChaseComponent

ChaseComponent.AIChaseComponent.AIChaseComponent.AIChaseComponent.Entity.AIChaseComponent.AIChaseComponent.AIChaseComponent.AIChaseComponent
```

``tokenizer.pad_token = tokenizer.eos_token ``
이렇게 되어있던 것을 다음과 같이 바꾼다.
``tokenizer.add_special_tokens({'pad_token': '[PAD]'}) model.resize_token_embeddings(len(tokenizer))``

=> 근데 이랬더니

```generated_code :
local currentTargetEntity = self.Entity.AI.AI.AI.AI.AI.AI.
AI.AI.AI.AI.AI.AI.AI.AI.AI.AI.AI.AI.AI.AI.AI
```

이렇게 됐다.

``repetition_penalty=1.2,``

생성할 때 저 파라미터를 넣었더니 반복해서 나오는 현상이 없어졌다.

``train_lua_v1.txt``로 학습을 했더니 ``prompt = "function add(a,b)\n return a + `` 를 했더니 b end로 답변. 원래는 b로 답했던 것과 비교하여 학습이 진행된 것을 알 수 있음.

**기존 모델과 학습 모델 비교는 compare.ipynb 참고**

In [ ]:
print(autocomplete_lua_function("local currentTargetEntity = self.Entity.AICh"))

generated_code : 
local currentTargetEntity = self.Entity.AIChaseComponent:GetCurrentTarget()
if currentTargetEntity == nil thenreturn
end
self.Entity.LookAtComponent:SetLookAt(currentTargetEntity)
```

aseComponent:GetCurrentTarget()
if currentTargetEntity == nil thenreturn
end
self.Entity.LookAtComponent:SetLookAt(currentTargetEntity)
```


In [ ]:
print(autocomplete_lua_function("# 둘이 더하는 lua 함수를 작성 \nfunction add(a,b)\n return a + "))

generated_code : 
# 둘이 더하는 lua 함수를 작성 
function add(a,b)
 return a +  b
end
local sum = _LuaService:RunFunctionAndWait("add",10,20)log (sum)
```

b
end
local sum = _LuaService:RunFunctionAndWait("add",10,20)log (sum)
```


# 학습된 tokenizer와 model 저장

In [ ]:
model = model.merge_and_unload()  # LoRA를 베이스에 병합
model.save_pretrained("저장하고 싶은 주소", safe_serialization=True)
tokenizer.save_pretrained("저장하고 싶은 주소", safe_serialization=True)

# Hugging Face 모델 올리기

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Repository 생성 & model upload
REPO_NAME = "maplestoryworlds-lua-api-finetune"
AUTH_TOKEN = "Hugging Face write api key" # <https://huggingface.co/settings/token>

## Upload to Huggingface Hub
model.push_to_hub(
    REPO_NAME,
    use_temp_dir=True,
    use_auth_token=AUTH_TOKEN
)
tokenizer.push_to_hub(
    REPO_NAME,
    use_temp_dir=True,
    use_auth_token=AUTH_TOKEN
)